In [ ]:
!pip install eynollah==0.9.2

In [ ]:
# this might need session restart
!apt install imagemagick

In [ ]:
# download inference-only layout models from zenodo
!wget https://zenodo.org/records/21381102/files/models_inference_layout_v0_9_1.zip?download=1

In [ ]:
!unzip /content/models_inference_layout_v0_9_1.zip

In [ ]:
!cd /content

In [ ]:
!mkdir -p outputs

In [ ]:
# change image to your own here
!wget https://github.com/qurator-spk/eynollah/raw/refs/heads/main/tests/resources/kant_aufklaerung_1784_0020.tif -O image.tif

In [ ]:
!eynollah layout -m /content/models_eynollah -i /content/image.tif -o /content/outputs -fl

In [ ]:
# visualize the pixel segmentation masks
import xml.etree.ElementTree as ET
from PIL import Image, ImageDraw


def draw_pagexml_segmentation_pil(image_path, xml_path, output_path):
    image = Image.open(image_path).convert("RGB")
    draw = ImageDraw.Draw(image)

    tree = ET.parse(xml_path)
    root = tree.getroot()
    ns = {"pc": "http://schema.primaresearch.org/PAGE/gts/pagecontent/2019-07-15"}


    for region in root.findall(".//pc:TextRegion", ns):
        coords = region.find("pc:Coords", ns)
        if coords is not None:
            points = coords.attrib["points"]
            pts = [tuple(map(int, p.split(","))) for p in points.split()]
            draw.polygon(pts, outline=(0, 0, 255), width=1)


    for line in root.findall(".//pc:TextLine", ns):
        coords = line.find("pc:Coords", ns)
        if coords is not None:
            points = coords.attrib["points"]
            pts = [tuple(map(int, p.split(","))) for p in points.split()]
            draw.polygon(pts, outline=(0, 206, 209), width=1)


    for line in root.findall(".//pc:ImageRegion", ns):
        coords = line.find("pc:Coords", ns)
        if coords is not None:
            points = coords.attrib["points"]
            pts = [tuple(map(int, p.split(","))) for p in points.split()]
            draw.polygon(pts, outline=(0, 255, 0), width=1)


    for line in root.findall(".//pc:SeparatorRegion", ns):
        coords = line.find("pc:Coords", ns)
        if coords is not None:
            points = coords.attrib["points"]
            pts = [tuple(map(int, p.split(","))) for p in points.split()]
            draw.polygon(pts, outline=(255, 20, 147), width=1)


    for line in root.findall(".//pc:TableRegion", ns):
        coords = line.find("pc:Coords", ns)
        if coords is not None:
            points = coords.attrib["points"]
            pts = [tuple(map(int, p.split(","))) for p in points.split()]
            draw.polygon(pts, outline=(255, 165, 0), width=1)


    image.save(output_path)
    print(f"Segmentation visualization saved to {output_path}")


file_name = "image"
file_ext = "tif"


draw_pagexml_segmentation_pil(
    f"/content/{file_name}.{file_ext}",
    f"/content/outputs/{file_name}.xml",
    f"segmentation_{file_name}.png",
)

from IPython.display import Image
Image(filename= f"/content/segmentation_{file_name}.png")